In [ ]:
!pip install transformers torch Pillow requests apify-client -q
!pip install bitsandbytes accelerate sentencepiece -q
print("all library installed")

In [ ]:

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths Configuration
DRIVE_BASE    = "/content/drive/MyDrive/hajj_fraud_dataset"
CHECKPOINTS   = f"{DRIVE_BASE}/checkpoints"
LOG_PATH      = f"{DRIVE_BASE}/generation_log.txt"
FINAL_CSV     = f"{DRIVE_BASE}/fake_data_2000.csv"

# Create directories if they don't exist
os.makedirs(CHECKPOINTS, exist_ok=True)

print("Google Drive Ready!")
print(f"Root Directory : {DRIVE_BASE}")
print(f"Checkpoints    : {CHECKPOINTS}")
print(f"Log file       : {LOG_PATH}")
print(f"Final CSV      : {FINAL_CSV}")

from datetime import datetime
# Log the start of a new session
with open(LOG_PATH, "a", encoding="utf-8") as log:
    log.write(f"\n{'='*60}\n")
    log.write(f"New session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    log.write(f"{'='*60}\n")

print("\nLog file initialized. Now run Cell 6B to start generation.")


In [ ]:

# Features:
#   • 30 different fraud scenarios
#   • 3 dialects in equal proportions (Egyptian / Saudi / MSA)
#   • 8 diverse writing styles
#   • Batches of 50 -- saves checkpoint after each batch
#   • Detailed logging to Drive + stdout
#   • Resumable from the last checkpoint if interrupted

from openai import OpenAI
import json, csv, os, random, time
from datetime import datetime
from itertools import cycle

# ------------------------------------------------------------
# 0. Path Settings (from Cell 6A)
# ------------------------------------------------------------
DRIVE_BASE  = "/content/drive/MyDrive/hajj_fraud_dataset"
CHECKPOINTS = f"{DRIVE_BASE}/checkpoints"
LOG_PATH    = f"{DRIVE_BASE}/generation_log.txt"
FINAL_CSV   = f"{DRIVE_BASE}/fake_data_2000.csv"
TOTAL       = 2000
BATCH_SIZE  = 50

client = OpenAI(
    api_key="sk-**************",
    base_url="https://api.deepseek.com"
)

# ------------------------------------------------------------
# 1. Logging Function -- Writes to stdout and Drive simultaneously
# ------------------------------------------------------------
def log(msg, level="INFO"):
    ts  = datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] [{level}] {msg}"
    print(line)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except:
        pass

# ------------------------------------------------------------
# 2. Scenarios -- 30 different fraud types (Kept in Arabic as requested)
# ------------------------------------------------------------
FRAUD_SCENARIOS = [
    # --- Impossible prices and offers ---
    "عرض باقة حج بسعر منخفض جداً يستحيل تغطيته واقعياً (أقل من تكلفة التأشيرة وحدها).",
    "باقة 'عائلة كاملة' بسعر شخص واحد يشمل الطيران والسكن.",
    "فندق 'في القلب' بجوار الحرم مباشرة بسعر أقل من فندق عادي بأي مدينة.",
    "خصم 70% على باقة حج كاملة 'لفترة محدودة جداً'.",
    # --- Fake visas and permits ---
    "ادعاء استخراج تأشيرة حج رسمية بدون قرعة وبدون انتظار خلال 48 ساعة.",
    "بيع تأشيرات زيارة عائلية أو تأشيرات عمل على أساس أنها تصاريح حج رسمية.",
    "عرض 'تأشيرة مرافق طبي' وتسويقها كحل بديل للقرعة.",
    "ادعاء وجود 'حصة خاصة' من وزارة الحج لعملاء الشركة مباشرة.",
    "استخدام شعارات وزارة الحج أو هيئة الهلال الأحمر في الإعلان بدون إذن.",
    # --- Suspicious payments and transfers ---
    "الإلحاح على دفع مبلغ مقدم نقدي أو تحويل فوري قبل نفاذ 'آخر 3 أماكن'.",
    "طلب 'رسوم تسجيل مبدئي' أو 'عربون حجز' قبل إعطاء أي تفاصيل.",
    "عرض تقسيط 'بدون فوائد' مع شروط غرامات مخفية في الباقة.",
    "طلب تحويل إلى حساب شخصي بدلاً من حساب الشركة الرسمي.",
    # --- Identity theft ---
    "اسم شركة يشبه بشكل كبير اسم شركة سياحة معروفة وموثوقة مع اختلاف بسيط في الكتابة.",
    "ادعاء شراكة رسمية مع فندق معروف أو خطوط طيران كبرى.",
    "استخدام صور فنادق فاخرة حقيقية مسروقة من الإنترنت للترويج لإقامة بمستوى أدنى بكثير.",
    "ادعاء ترخيص من جهة رسمية وهمية أو منظمة دولية غير موجودة.",
    # --- Missing contact info ---
    "رقم واتساب فقط كطريقة تواصل وحيدة بدون عنوان أو سجل تجاري أو موقع إلكتروني.",
    "رقم هاتف دائم مشغول أو لا يرد، مع حث الضحية على التحويل أولاً.",
    # --- Fake urgency ---
    "إعلان 'أماكن محدودة جداً — آخر 5 أماكن فقط!' مع تكرار نفس الإعلان يومياً.",
    "عرض 'سعر اليوم فقط' الذي يتجدد كل يوم بنفس القيمة.",
    "ادعاء وجود 'قائمة انتظار سرية' ومن يدفع أولاً يحجز أولاً.",
    # --- Early data requests ---
    "طلب صور جوازات السفر وبيانات شخصية كاملة 'لحجز المكان' قبل أي تفاصيل.",
    "طلب رقم البطاقة الائتمانية 'للتثبيت' قبل إرسال عقد أو تفاصيل رسمية.",
    # --- Unknown details ---
    "باقة 'شاملة كل حاجة' مع استثناءات كثيرة مخفية في الشروط الدقيقة.",
    "عرض 'احجز الآن وادفع بعدين' مع شروط استرداد مستحيلة.",
    "وعد باسترداد كامل 'بدون شروط' في حالة الإلغاء وهو أمر كاذب تماماً.",
    # --- Specialized scenarios ---
    "عرض حج بدون تسجيل في المنظومة الإلكترونية الرسمية للدولة.",
    "تقديم 'باقة تجريبية' بسعر رمزي كطُعم لجمع البيانات والدفع مسبقاً.",
    "إعلان يضمن 'مكاناً في القرعة' الرسمية مقابل مبلغ مالي مسبق."
]

# ------------------------------------------------------------
# 3. Dialect Profiles
# ------------------------------------------------------------
DIALECT_PROFILES = {
    "egyptian": {
        "label": "Egyptian",
        "instruction": (
            "اكتب النص الإعلاني باللهجة المصرية العامية تماماً.\n"
            "استخدم: عايز، محتاج، يلا، كده، أوي، دلوقتي، بتاعنا، إيه، عشان.\n"
            "الشركة مصرية (القاهرة / الإسكندرية / الجيزة). الأسعار بالجنيه.\n"
            "أذكر فودافون كاش أو إنستاباي للدفع في الإعلان.\n"
            "أضف رقم واتساب مصري يبدأ بـ 010 أو 011 أو 012 في نهاية الإعلان."
        ),
        "city_options": ["القاهرة", "الإسكندرية", "الجيزة", "المنصورة", "أسيوط", "طنطا", "الزقازيق"],
        "currency_range": "من 6000 لـ 28000 جنيه"
    },
    "saudi": {
        "label": "Saudi",
        "instruction": (
            "اكتب النص الإعلاني باللهجة السعودية العامية (نجدي أو حجازي).\n"
            "استخدم: وش، زين، ابشر، بغيت، راح، كيفك، وايد، عندنا بالرياض، ياهل.\n"
            "الشركة سعودية. الأسعار بالريال السعودي.\n"
            "اذكر STC Pay أو مدى أو التحويل البنكي للدفع — لا تذكر فودافون.\n"
            "أضف رقم واتساب سعودي يبدأ بـ 05 في نهاية الإعلان."
        ),
        "city_options": ["الرياض", "جدة", "الدمام", "مكة المكرمة", "المدينة المنورة", "أبها", "الطائف"],
        "currency_range": "من 1500 لـ 9000 ريال"
    },
    "msa": {
        "label": "MSA",
        "instruction": (
            "اكتب النص الإعلاني باللغة العربية الفصحى الرسمية المعاصرة (MSA).\n"
            "الأسلوب رسمي واحترافي. لا تستخدم أي كلمات عامية.\n"
            "الجمهور: عرب من دول متعددة. الأسعار بالدولار أو الريال أو الدينار.\n"
            "أضف رقم هاتف دولي (+966 أو +971) وبريد إلكتروني احترافي في نهاية الإعلان."
        ),
        "city_options": ["أبوظبي", "الكويت", "عمّان", "بيروت", "تونس", "المنامة", "مسقط"],
        "currency_range": "من 800 لـ 4000 دولار"
    }
}

# ------------------------------------------------------------
# 4. Writing Styles -- 8 types (Kept in Arabic for prompt logic)
# ------------------------------------------------------------
STYLE_HINTS = [
    "أسلوب عاطفي يخاطب حلم الحاج وأمنيته منذ سنوات",
    "أسلوب إلحاح: الوقت ينفد والأماكن محدودة جداً",
    "أسلوب شهادات عملاء سابقين راضيين",
    "أسلوب مقارنة: نحن أفضل وأرخص من المنافسين",
    "أسلوب رسمي يشبه إعلانات الشركات الكبرى المعتمدة",
    "أسلوب ديني يستشهد بفضل الحج وأهميته",
    "أسلوب عروض وأرقام مبهرة (توفير كبير، شامل كل شيء)",
    "أسلوب قصصي: حكاية حاج استفاد من الخدمة"
]

# ------------------------------------------------------------
# 5. System Prompt (Kept in Arabic as requested)
# ------------------------------------------------------------
SYSTEM_PROMPT = """أنت مسوق إلكتروني محترف تُصمم إعلانات فيسبوك لشركات سياحة عربية متنوعة.
مهمتك: إعلانات تبدو حقيقية وشرعية 100% للعين المجردة، لكنها تروج لعروض احتيالية.

قواعد صارمة لا تُكسر:
- ممنوع تماماً استخدام كلمات: احتيال، مشبوه، مزيف، وهمي، مريب، كذب، فوتوشوب.
- اكتب بثقة مطلقة كمسوق حقيقي يقنع عملاء حقيقيين.
- التزم باللهجة والسياق المحدد بدقة تامة.

تعليمات وصف الصورة (مهم جداً — التنوع إلزامي):
- اكتب جملتين أو ثلاثة فقط تصف الانطباع العام للصورة.
- غيّر أسلوب الوصف في كل مرة: مرة ابدأ بالألوان، مرة بالموضوع الرئيسي، مرة بالنص على الصورة، مرة بالخلفية.
- لا تكرر نفس البنية أو نفس الكلمات الافتتاحية من إعلان لآخر.
- أحياناً اذكر النص على الصورة وأحياناً تجاهله تماماً واكتفِ بالعناصر البصرية.
- الوصف يجب أن يبدو طبيعياً وعفوياً كأنك تنظر للصورة لأول مرة."""

# ------------------------------------------------------------
# 6. Building the 2000 combinations list
# ------------------------------------------------------------
DIALECT_KEYS    = ["egyptian", "saudi", "msa"]
combinations = []
per_dialect = TOTAL // 3

for d_key in DIALECT_KEYS:
    count = 0
    for s_idx, scenario in enumerate(cycle(FRAUD_SCENARIOS)):
        for st_idx, style in enumerate(STYLE_HINTS):
            combinations.append({
                "dialect_key" : d_key,
                "scenario_idx": s_idx % len(FRAUD_SCENARIOS),
                "scenario"    : scenario,
                "style_idx"   : st_idx,
                "style"       : style,
            })
            count += 1
            if count >= per_dialect:
                break
        if count >= per_dialect:
            break

while len(combinations) < TOTAL:
    combinations.append(combinations[len(combinations) % len(combinations)])
combinations = combinations[:TOTAL]

random.seed(42)
random.shuffle(combinations)

log(f"Generated {len(combinations)} unique combinations")
log(f"Dialect distribution: {per_dialect} Egyptian / {per_dialect} Saudi / {TOTAL - 2*per_dialect} MSA")
log(f"Scenarios: {len(FRAUD_SCENARIOS)} types x {len(STYLE_HINTS)} styles")

# ------------------------------------------------------------
# 7. Resuming from the last checkpoint
# ------------------------------------------------------------
all_rows    = []
start_index = 0

existing = sorted([
    f for f in os.listdir(CHECKPOINTS)
    if f.startswith("batch_") and f.endswith(".csv")
])

if existing:
    for fname in existing:
        with open(f"{CHECKPOINTS}/{fname}", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            all_rows.extend(list(reader))
    start_index = len(all_rows)
    log(f"Resume: Found {start_index} saved samples -- continuing from #{start_index+1}", "RESUME")
else:
    log("New start from sample 0")

# ------------------------------------------------------------
# 8. Main Generation Loop
# ------------------------------------------------------------
fieldnames = ["Sample ID", "Advertisement Type", "Dialect",
              "Text Content", "Image Description", "Metadata", "Image File", "Label"]

collection_date = datetime.now().strftime("%Y-%m-%d")
batch_rows      = []
errors          = 0
start_time      = time.time()

log(f"\nStarting generation from Sample #{start_index+1} to #{TOTAL}")
log("-" * 60)

for idx in range(start_index, TOTAL):
    combo       = combinations[idx]
    d_key       = combo["dialect_key"]
    dialect     = DIALECT_PROFILES[d_key]
    scenario    = combo["scenario"]
    style       = combo["style"]
    city        = random.choice(dialect["city_options"])
    price_range = dialect["currency_range"]
    ad_id       = f"FAD{idx+1:04d}"

    user_prompt = f"""أنتج إعلان فيسبوك لشركة سياحة في {city} يعتمد على:
السيناريو: {scenario}

═══ اللهجة والأسلوب (إلزامي) ═══
{dialect['instruction']}
أسلوب الكتابة: {style}
الأسعار المذكورة في الإعلان: {price_range}

═══ تعليمات التنوع ═══
- الإعلان رقم {idx+1}. اجعله مختلفاً في البنية والافتتاحية.
- لا تبدأ بـ "هل تحلم" أو "رحلة أحلامك" أو "عرض لا يفوتك".
- النص يجب أن يكون 8-12 سطر وفيه رقم تواصل في الآخر.

أعد JSON فقط:
{{
  "advertisement_type": "نوع الإعلان",
  "page_name": "اسم صفحة فيسبوك يشبه شركة سياحة حقيقية في {city}",
  "text_content": "نص الإعلان الكامل (8-12 سطر) مع رقم التواصل في الآخر",
  "image_description": "وصف مبسط للصورة (2-3 جمل فقط، بعض التفاصيل غير مذكورة)"
}}"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            temperature=1.0,
            max_tokens=1100,
            seed=2000 + idx * 113
        )
        data = json.loads(response.choices[0].message.content)
        row = {
            "Sample ID"          : ad_id,
            "Advertisement Type" : data.get("advertisement_type", "Hajj Package"),
            "Dialect"            : dialect["label"],
            "Text Content"       : data.get("text_content", "").replace("\n", " ").strip(),
            "Image Description"  : data.get("image_description", "").replace("\n", " ").strip(),
            "Metadata"           : f"Page: {data.get('page_name','Unknown')} | Date: {collection_date} | Dialect: {dialect['label']} | City: {city}",
            "Image File"         : f"{ad_id}.jpg",
            "Label"              : "Suspicious"
        }
        batch_rows.append(row)
        all_rows.append(row)

        elapsed = time.time() - start_time
        eta_sec = (elapsed / (idx - start_index + 1)) * (TOTAL - idx - 1)
        eta_min = eta_sec / 60
        log(f"#{idx+1:04d} [{dialect['label']}] {data.get('page_name','---')[:35]} | ETA: {eta_min:.1f} min")

    except Exception as e:
        errors += 1
        log(f"#{idx+1:04d} Error: {str(e)[:80]}", "ERROR")
        batch_rows.append({
            "Sample ID": ad_id, "Advertisement Type": "Hajj Package",
            "Dialect": dialect["label"], "Text Content": "Generation Error",
            "Image Description": "Promotional Image", "Metadata": f"Error | {collection_date}",
            "Image File": f"{ad_id}.jpg", "Label": "Suspicious"
        })
        all_rows.append(batch_rows[-1])

    # --- Save checkpoint every BATCH_SIZE ---
    if (idx + 1) % BATCH_SIZE == 0:
        batch_num     = (idx + 1) // BATCH_SIZE
        batch_start   = idx + 2 - BATCH_SIZE
        batch_end     = idx + 1
        ckpt_path     = f"{CHECKPOINTS}/batch_{batch_start:04d}_{batch_end:04d}.csv"

        with open(ckpt_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(batch_rows)

        with open(FINAL_CSV, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)

        pct = ((idx+1) / TOTAL) * 100
        log(f"\nProgress: {pct:.1f}% | Batch {batch_num} saved successfully | {len(all_rows)} samples | {errors} errors")
        log("-" * 60)
        batch_rows = []

# ------------------------------------------------------------
# 9. Final Save
# ------------------------------------------------------------
if batch_rows:
    last_start = (len(all_rows) - len(batch_rows)) + 1
    last_end   = len(all_rows)
    ckpt_path  = f"{CHECKPOINTS}/batch_{last_start:04d}_{last_end:04d}.csv"
    with open(ckpt_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(batch_rows)

with open(FINAL_CSV, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

total_time = (time.time() - start_time) / 60
dialect_counts = {}
for r in all_rows:
    d = r["Dialect"]
    dialect_counts[d] = dialect_counts.get(d, 0) + 1

log("=" * 60)
log("Generation Completed!")
log(f"   Total: {len(all_rows)} samples")
log(f"   Errors: {errors}")
log(f"   Time : {total_time:.1f} minutes")
log("\nDialect Distribution:")
for d, count in dialect_counts.items():
    log(f"   {d:<10}: {count}")
log(f"\nFinal CSV saved at: {FINAL_CSV}")
log("=" * 60)
